# Effects and Realism: Modeling Technical Variation

Real spatial transcriptomics data contains various sources of **technical variation** that affect downstream analysis. PointillSim provides sophisticated tools to simulate these effects, enabling more realistic benchmarks.

## What You'll Learn

1. **Batch Effects** - Systematic variation between samples/FOVs
2. **Technical Noise** - Amplification, optical, and focus variations
3. **Background Noise** - False positive transcript detections
4. **Dropout Model** - Gene-specific detection failures
5. **Combining Effects** - Creating realistic noisy datasets

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.collections import PatchCollection
import pandas as pd

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    DistanceBasedRule,
    plot_fov,
)

# Import effects module
from pointillsim.effects import (
    BatchEffectModel,
    TechnicalNoise,
    BackgroundNoise,
    DropoutModel,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Create a reference tissue and FOV for demonstrations
n_cell_types = 5
n_genes = 50
frame_size = 600

# Create tissue with clear markers
tissue = TissueCellTypes()
tissue.generate_types_and_markers(
    n_genes=n_genes,
    n_cell_types=n_cell_types,
    expected_level=15.0,
    concentration=0.9,
)

cell_props = CellTypesProperties(n_cell_types=n_cell_types)

# Create FOV distribution with structures
fov_dist = FOVDistribution(
    frame_size=frame_size,
    background_element=lambda: FrameWideElement(
        frame_size=frame_size,
        tipical_cell_spacing=18,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types,
            list_N=[0, 1],
            proportions=[0.6, 0.4]
        )
    ),
    other_elements=[
        lambda: VacuolatedStructure(
            frame_size=frame_size,
            scale=80,
            hole_scale_factor=0.4,
            tipical_cell_spacing=12,
            rules=DistanceBasedRule(
                n_cell_types=n_cell_types,
                inner_type=2,
                outer_type=3,
            )
        )
    ],
    elements_frequency=[1.0],
    attempts_at_elements=4,
)

# Generate reference FOV
np.random.seed(42)
fov_ref = fov_dist.generate_fov()
cell_props.apply(fov_ref)

print(f"Reference FOV: {fov_ref.n_cells} cells, {n_genes} genes")

---
## 1. Batch Effects

**Batch effects** are systematic differences between experimental batches (slides, experiments, days). They're a major source of unwanted variation in real data.

The `BatchEffectModel` simulates:
- **Global shifts**: All genes affected similarly
- **Gene-specific shifts**: Each gene responds differently to batch
- **Multiplicative/additive effects**: Different modes of batch variation

In [ ]:
# Create a batch effect model
batch_model = BatchEffectModel(
    n_genes=n_genes,
    global_scale_std=0.3,      # 30% variation in global expression level
    gene_scale_std=0.2,        # 20% gene-specific variation
    global_offset_std=0.5,     # Additive global offset
    gene_offset_std=0.3,       # Additive gene-specific offset
)

print("BatchEffectModel created")
print(f"  Global scale range: {batch_model.global_scale_std:.1%} std")
print(f"  Gene-specific scale range: {batch_model.gene_scale_std:.1%} std")

In [ ]:
# Generate multiple 'batches' from the same FOV
n_batches = 5

fig, axes = plt.subplots(2, n_batches, figsize=(4*n_batches, 8))

all_dots = []
batch_labels = []

for batch_id in range(n_batches):
    # Generate batch-specific effects
    batch_effects = batch_model.sample_batch_effects()
    
    # Apply batch effects to tissue expression
    tissue_batch = TissueCellTypes()
    tissue_batch.gene_expression_by_type = batch_model.apply_batch_effects(
        tissue.gene_expression_by_type, 
        batch_effects
    )
    
    # Generate observations
    hybiss = HybISS_Setup(tissue_batch)
    hybiss.observe_dots(fov_ref)
    dots_df = hybiss.make_pandas_df()
    dots_df['batch'] = batch_id
    all_dots.append(dots_df)
    
    # Visualize transcript dots
    ax = axes[0, batch_id]
    ax.scatter(dots_df['x'], dots_df['y'], s=1, alpha=0.4, c='darkred')
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'Batch {batch_id+1}\n({len(dots_df):,} dots)')
    
    # Gene expression distribution
    ax = axes[1, batch_id]
    gene_counts = dots_df['gene'].value_counts()
    ax.bar(range(min(20, len(gene_counts))), gene_counts.values[:20], alpha=0.7)
    ax.set_xlabel('Gene rank')
    ax.set_ylabel('Count')
    ax.set_title('Top 20 genes')

plt.suptitle('Batch Effects: Same Tissue, Different Technical Conditions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Combine all dots
combined_dots = pd.concat(all_dots, ignore_index=True)
print(f"\nTotal dots per batch:")
print(combined_dots.groupby('batch').size())

In [ ]:
# Visualize batch effects on gene expression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gene counts across batches
ax = axes[0]
gene_by_batch = combined_dots.groupby(['batch', 'gene']).size().unstack(fill_value=0)
# Select top 10 most variable genes
gene_var = gene_by_batch.var()
top_var_genes = gene_var.nlargest(10).index

x = np.arange(n_batches)
width = 0.08
colors = plt.cm.Set2(np.linspace(0, 1, 10))

for i, gene in enumerate(top_var_genes):
    ax.bar(x + i*width, gene_by_batch[gene].values, width, 
           label=gene, color=colors[i], alpha=0.8)

ax.set_xlabel('Batch')
ax.set_ylabel('Transcript Count')
ax.set_title('Top 10 Most Variable Genes Across Batches')
ax.set_xticks(x + width*4.5)
ax.set_xticklabels([f'Batch {i+1}' for i in range(n_batches)])
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

# Correlation between batches
ax = axes[1]
batch_corr = gene_by_batch.T.corr()
im = ax.imshow(batch_corr, cmap='RdYlBu_r', vmin=0.5, vmax=1.0)
ax.set_xticks(range(n_batches))
ax.set_yticks(range(n_batches))
ax.set_xticklabels([f'B{i+1}' for i in range(n_batches)])
ax.set_yticklabels([f'B{i+1}' for i in range(n_batches)])
ax.set_title('Gene Expression Correlation\nBetween Batches')
for i in range(n_batches):
    for j in range(n_batches):
        ax.text(j, i, f'{batch_corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='Pearson r')

plt.tight_layout()
plt.show()

---
## 2. Technical Noise

**Technical noise** models sources of variation within a single FOV:

- **Amplification efficiency**: Variation in PCR/probe binding
- **Optical field uniformity**: Vignetting effects (edges darker)
- **Focus-dependent efficiency**: Variation across z-planes

In [ ]:
# Create technical noise model
tech_noise = TechnicalNoise(
    amplification_cv=0.15,          # 15% coefficient of variation in amplification
    optical_vignetting_strength=0.3, # 30% reduction at corners
    focus_variation_strength=0.1,    # 10% focus-related variation
)

print("TechnicalNoise model created")
print(f"  Amplification CV: {tech_noise.amplification_cv:.0%}")
print(f"  Vignetting strength: {tech_noise.optical_vignetting_strength:.0%}")

In [ ]:
# Visualize optical vignetting effect
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a grid of points
x_grid = np.linspace(0, frame_size, 50)
y_grid = np.linspace(0, frame_size, 50)
xx, yy = np.meshgrid(x_grid, y_grid)
positions = np.column_stack([xx.ravel(), yy.ravel()])

# Calculate vignetting factor
vignetting = tech_noise.compute_vignetting_factor(positions, frame_size)

ax = axes[0]
im = ax.scatter(positions[:, 0], positions[:, 1], c=vignetting, 
                cmap='viridis', s=80, marker='s')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Optical Vignetting\n(Detection Efficiency by Position)')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.colorbar(im, ax=ax, label='Efficiency')

# Apply to real FOV - no noise
np.random.seed(42)
hybiss_clean = HybISS_Setup(tissue)
hybiss_clean.observe_dots(fov_ref)
dots_clean = hybiss_clean.make_pandas_df()

ax = axes[1]
ax.scatter(dots_clean['x'], dots_clean['y'], s=1, alpha=0.3, c='darkblue')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Without Vignetting\n({len(dots_clean):,} dots)')

# Apply vignetting by probabilistic dropout
np.random.seed(42)
positions_dots = dots_clean[['x', 'y']].values
efficiency = tech_noise.compute_vignetting_factor(positions_dots, frame_size)
keep_mask = np.random.random(len(dots_clean)) < efficiency
dots_vignetted = dots_clean[keep_mask].copy()

ax = axes[2]
ax.scatter(dots_vignetted['x'], dots_vignetted['y'], s=1, alpha=0.3, c='darkred')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'With Vignetting\n({len(dots_vignetted):,} dots)')

plt.suptitle('Optical Vignetting: Reduced Detection at FOV Edges', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nDots lost to vignetting: {len(dots_clean) - len(dots_vignetted)} ({100*(1-len(dots_vignetted)/len(dots_clean)):.1f}%)")

---
## 3. Background Noise

**Background noise** represents false positive detections from:
- Autofluorescence
- Ambient RNA
- Non-specific probe binding

These appear as randomly distributed dots unrelated to cell positions.

In [ ]:
# Create background noise model
bg_noise = BackgroundNoise(
    density=0.001,  # dots per square pixel
    gene_distribution='uniform',  # or 'proportional' to true expression
)

# Expected background dots
expected_bg = bg_noise.density * frame_size * frame_size
print(f"BackgroundNoise model created")
print(f"  Density: {bg_noise.density:.4f} dots/pixel²")
print(f"  Expected background dots per FOV: {expected_bg:.0f}")

In [ ]:
# Generate background noise
bg_dots = bg_noise.generate(
    frame_size=frame_size,
    gene_names=tissue.gene_names[:10],  # Use first 10 genes
)

# Combine with real signal
combined = pd.concat([
    dots_clean.assign(source='signal'),
    bg_dots.assign(source='background', cell=-1)
], ignore_index=True)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Signal only
ax = axes[0]
ax.scatter(dots_clean['x'], dots_clean['y'], s=2, alpha=0.4, c='blue', label='Signal')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Signal Only\n({len(dots_clean):,} dots)')

# Background only
ax = axes[1]
ax.scatter(bg_dots['x'], bg_dots['y'], s=3, alpha=0.6, c='orange', label='Background')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Background Only\n({len(bg_dots):,} dots)')

# Combined
ax = axes[2]
signal_mask = combined['source'] == 'signal'
ax.scatter(combined.loc[signal_mask, 'x'], combined.loc[signal_mask, 'y'], 
           s=2, alpha=0.4, c='blue', label='Signal')
ax.scatter(combined.loc[~signal_mask, 'x'], combined.loc[~signal_mask, 'y'], 
           s=3, alpha=0.6, c='orange', label='Background')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Combined\n({len(combined):,} total dots)')
ax.legend(loc='upper right', markerscale=3)

plt.suptitle('Background Noise: False Positive Detections', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nSignal-to-background ratio: {len(dots_clean)/len(bg_dots):.1f}")

---
## 4. Dropout Model

**Gene-specific dropout** models the phenomenon where some genes are more reliably detected than others, even at similar expression levels.

Causes include:
- Probe design quality
- Secondary structure
- GC content

In [ ]:
# Create dropout model
dropout = DropoutModel(
    n_genes=n_genes,
    mean_dropout_rate=0.15,    # Average 15% dropout
    dropout_rate_std=0.1,      # Variation across genes
    expression_dependent=True,  # Lower expression = higher dropout
)

print("DropoutModel created")
print(f"  Mean dropout rate: {dropout.mean_dropout_rate:.0%}")
print(f"  Dropout rate range: {dropout.gene_dropout_rates.min():.0%} - {dropout.gene_dropout_rates.max():.0%}")

In [ ]:
# Visualize gene-specific dropout rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dropout rates by gene
ax = axes[0]
sorted_idx = np.argsort(dropout.gene_dropout_rates)
colors = plt.cm.RdYlGn_r(dropout.gene_dropout_rates[sorted_idx])
ax.barh(range(n_genes), dropout.gene_dropout_rates[sorted_idx], color=colors)
ax.axvline(dropout.mean_dropout_rate, color='red', linestyle='--', 
           label=f'Mean: {dropout.mean_dropout_rate:.0%}')
ax.set_xlabel('Dropout Rate')
ax.set_ylabel('Gene (sorted)')
ax.set_title('Gene-Specific Dropout Rates')
ax.legend()

# Apply dropout to dots
np.random.seed(42)
dots_with_dropout = dropout.apply(dots_clean, gene_col='gene')

# Compare detection before/after
ax = axes[1]
gene_counts_before = dots_clean['gene'].value_counts()
gene_counts_after = dots_with_dropout['gene'].value_counts()

# Align indices
common_genes = gene_counts_before.index[:20]
x = np.arange(len(common_genes))
width = 0.35

ax.bar(x - width/2, [gene_counts_before.get(g, 0) for g in common_genes], 
       width, label='Before dropout', alpha=0.8)
ax.bar(x + width/2, [gene_counts_after.get(g, 0) for g in common_genes], 
       width, label='After dropout', alpha=0.8)
ax.set_xlabel('Gene')
ax.set_ylabel('Count')
ax.set_title('Effect of Dropout on Gene Counts')
ax.set_xticks(x)
ax.set_xticklabels(common_genes, rotation=45, ha='right', fontsize=8)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nDots before dropout: {len(dots_clean):,}")
print(f"Dots after dropout: {len(dots_with_dropout):,}")
print(f"Overall dropout: {100*(1-len(dots_with_dropout)/len(dots_clean)):.1f}%")

---
## 5. Combining Effects: Realistic Noisy Datasets

Real data contains **all these effects simultaneously**. Let's create a complete pipeline that applies multiple sources of variation.

In [ ]:
def generate_realistic_fov(
    fov_dist,
    tissue,
    cell_props,
    batch_model=None,
    tech_noise=None,
    bg_noise=None,
    dropout_model=None,
    seed=None,
):
    """
    Generate a realistic FOV with multiple sources of technical variation.
    
    Parameters
    ----------
    fov_dist : FOVDistribution
        Distribution to sample FOV from
    tissue : TissueCellTypes
        Reference tissue expression
    cell_props : CellTypesProperties
        Cell morphology properties
    batch_model : BatchEffectModel, optional
        Batch effect model
    tech_noise : TechnicalNoise, optional
        Technical noise model
    bg_noise : BackgroundNoise, optional
        Background noise model
    dropout_model : DropoutModel, optional
        Dropout model
    seed : int, optional
        Random seed
        
    Returns
    -------
    fov : FOV
        Generated FOV
    dots_df : pd.DataFrame
        Transcript dots with effects applied
    effects_log : dict
        Log of applied effects
    """
    if seed is not None:
        np.random.seed(seed)
    
    effects_log = {}
    
    # Generate FOV
    fov = fov_dist.generate_fov()
    cell_props.apply(fov)
    effects_log['n_cells'] = fov.n_cells
    
    # Apply batch effects to tissue
    tissue_effective = TissueCellTypes()
    if batch_model is not None:
        batch_effects = batch_model.sample_batch_effects()
        tissue_effective.gene_expression_by_type = batch_model.apply_batch_effects(
            tissue.gene_expression_by_type, batch_effects
        )
        effects_log['batch_global_scale'] = batch_effects.get('global_scale', 1.0)
    else:
        tissue_effective.gene_expression_by_type = tissue.gene_expression_by_type.copy()
    
    # Generate base observations
    hybiss = HybISS_Setup(tissue_effective)
    hybiss.observe_dots(fov)
    dots_df = hybiss.make_pandas_df()
    effects_log['dots_raw'] = len(dots_df)
    
    # Apply vignetting
    if tech_noise is not None:
        positions = dots_df[['x', 'y']].values
        efficiency = tech_noise.compute_vignetting_factor(positions, fov_dist.frame_size)
        keep_mask = np.random.random(len(dots_df)) < efficiency
        dots_df = dots_df[keep_mask].copy()
        effects_log['dots_after_vignetting'] = len(dots_df)
    
    # Apply dropout
    if dropout_model is not None:
        dots_df = dropout_model.apply(dots_df, gene_col='gene')
        effects_log['dots_after_dropout'] = len(dots_df)
    
    # Add background noise
    if bg_noise is not None:
        bg_dots = bg_noise.generate(
            frame_size=fov_dist.frame_size,
            gene_names=tissue.gene_names,
        )
        bg_dots['cell'] = -1  # Not associated with any cell
        dots_df = pd.concat([dots_df, bg_dots], ignore_index=True)
        effects_log['background_dots'] = len(bg_dots)
    
    effects_log['dots_final'] = len(dots_df)
    
    return fov, dots_df, effects_log

In [ ]:
# Create all effect models
batch_model_full = BatchEffectModel(
    n_genes=n_genes,
    global_scale_std=0.25,
    gene_scale_std=0.15,
)

tech_noise_full = TechnicalNoise(
    amplification_cv=0.1,
    optical_vignetting_strength=0.25,
)

bg_noise_full = BackgroundNoise(density=0.0005)

dropout_full = DropoutModel(
    n_genes=n_genes,
    mean_dropout_rate=0.12,
    dropout_rate_std=0.08,
)

print("Effect models configured for realistic simulation")

In [ ]:
# Generate clean vs realistic comparison
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Clean FOV (no effects)
np.random.seed(123)
fov_clean, dots_clean_new, log_clean = generate_realistic_fov(
    fov_dist, tissue, cell_props, seed=123
)

# Realistic FOV (all effects)
fov_real, dots_real, log_real = generate_realistic_fov(
    fov_dist, tissue, cell_props,
    batch_model=batch_model_full,
    tech_noise=tech_noise_full,
    bg_noise=bg_noise_full,
    dropout_model=dropout_full,
    seed=123,
)

# Row 1: Clean
ax = axes[0, 0]
ax.scatter(fov_clean.cell_centroids[:, 0], fov_clean.cell_centroids[:, 1],
           c=fov_clean.class_instance, cmap='Set1', s=20, alpha=0.7)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Clean: Cell Types')

ax = axes[0, 1]
ax.scatter(dots_clean_new['x'], dots_clean_new['y'], s=1, alpha=0.3, c='blue')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Clean: Transcripts\n({len(dots_clean_new):,} dots)')

ax = axes[0, 2]
gene_counts_clean = dots_clean_new['gene'].value_counts()
ax.bar(range(min(20, len(gene_counts_clean))), gene_counts_clean.values[:20], color='blue', alpha=0.7)
ax.set_xlabel('Gene rank')
ax.set_ylabel('Count')
ax.set_title('Clean: Gene Distribution')

ax = axes[0, 3]
ax.text(0.5, 0.5, f"Total dots: {log_clean['dots_final']:,}", 
        ha='center', va='center', fontsize=14, transform=ax.transAxes)
ax.set_title('Clean: Summary')
ax.axis('off')

# Row 2: Realistic
ax = axes[1, 0]
ax.scatter(fov_real.cell_centroids[:, 0], fov_real.cell_centroids[:, 1],
           c=fov_real.class_instance, cmap='Set1', s=20, alpha=0.7)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Realistic: Cell Types')

ax = axes[1, 1]
signal_mask = dots_real['cell'] >= 0
ax.scatter(dots_real.loc[signal_mask, 'x'], dots_real.loc[signal_mask, 'y'], 
           s=1, alpha=0.3, c='blue', label='Signal')
ax.scatter(dots_real.loc[~signal_mask, 'x'], dots_real.loc[~signal_mask, 'y'], 
           s=2, alpha=0.5, c='orange', label='Background')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Realistic: Transcripts\n({len(dots_real):,} dots)')
ax.legend(markerscale=3, loc='upper right')

ax = axes[1, 2]
gene_counts_real = dots_real['gene'].value_counts()
ax.bar(range(min(20, len(gene_counts_real))), gene_counts_real.values[:20], color='darkred', alpha=0.7)
ax.set_xlabel('Gene rank')
ax.set_ylabel('Count')
ax.set_title('Realistic: Gene Distribution')

ax = axes[1, 3]
summary_text = (
    f"Raw dots: {log_real['dots_raw']:,}\n"
    f"After vignetting: {log_real['dots_after_vignetting']:,}\n"
    f"After dropout: {log_real['dots_after_dropout']:,}\n"
    f"+ Background: {log_real['background_dots']:,}\n"
    f"─────────────\n"
    f"Final: {log_real['dots_final']:,}"
)
ax.text(0.5, 0.5, summary_text, ha='center', va='center', fontsize=11, 
        transform=ax.transAxes, family='monospace')
ax.set_title('Realistic: Effect Summary')
ax.axis('off')

plt.suptitle('Clean vs Realistic Simulation: Impact of Technical Effects', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Generate a batch of realistic FOVs to show variation
n_samples = 8

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for i, ax in enumerate(axes.flat):
    fov_i, dots_i, log_i = generate_realistic_fov(
        fov_dist, tissue, cell_props,
        batch_model=batch_model_full,
        tech_noise=tech_noise_full,
        bg_noise=bg_noise_full,
        dropout_model=dropout_full,
        seed=1000 + i,
    )
    
    ax.scatter(dots_i['x'], dots_i['y'], s=1, alpha=0.4, 
               c=np.where(dots_i['cell'] >= 0, 'darkblue', 'orange'))
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'Sample {i+1}\n{fov_i.n_cells} cells, {len(dots_i):,} dots')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Batch of Realistic FOVs: Natural Variation from Technical Effects', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated PointillSim's tools for simulating realistic technical variation:

| Effect | Class | Purpose |
|--------|-------|--------|
| **Batch Effects** | `BatchEffectModel` | Systematic variation between samples |
| **Technical Noise** | `TechnicalNoise` | Amplification, optical, focus variation |
| **Background** | `BackgroundNoise` | False positive detections |
| **Dropout** | `DropoutModel` | Gene-specific detection failures |

### Key Takeaways

1. **Batch effects** create systematic differences that must be corrected in analysis
2. **Vignetting** reduces detection at FOV edges - affects cell segmentation
3. **Background noise** adds false positives that contaminate expression estimates
4. **Dropout** disproportionately affects low-expressed genes
5. **Combined effects** create realistic data for robust benchmarking

### Next Steps

- **08_simulation_design.ipynb**: Design controlled experiments with covariates
- **09_validation_and_difficulty.ipynb**: Assess simulation difficulty and validate results